# 05 — Generate variants_ml.py

Runs the full feature pipeline for the 5 Indian variants, gets SIFT + PolyPhen from the Ensembl VEP REST API, runs the trained classifier, and writes `../backend/data/variants_ml.py`.

**Review workflow**: diff `variants_ml.py` against `variants.py` before replacing. The preserved fields (population_data, plain_language_summary, clinical_notes, common_name, uniprot_id) are carried over from the original file. The replaced fields are all computed feature values, severity, confidence, and tool_comparison.

**Run notebooks 01–04 first** — this notebook loads their output files.

In [1]:
import io
import json
import math
import time
import textwrap
import requests
import joblib
import numpy as np
from pathlib import Path
from Bio import AlignIO
from Bio.Align import substitution_matrices
from Bio.PDB import PDBParser, Superimposer

DATA_DIR   = Path('data')
MODELS_DIR = Path('models')
BACKEND_DATA = Path('../backend/data')

# Load trained artifacts
clf     = joblib.load(MODELS_DIR / 'hemoscope_rf_calibrated.pkl')
le      = joblib.load(MODELS_DIR / 'label_encoder.pkl')
imputer = joblib.load(MODELS_DIR / 'imputer.pkl')
with open(MODELS_DIR / 'feature_names.json') as f:
    FEATURE_COLS = json.load(f)

# Load pre-computed structural metadata (heme Fe coords, RMSD weights, etc.)
with open(DATA_DIR / 'structural_meta.json') as f:
    struct_meta = json.load(f)
RMSD_WEIGHTS      = np.array(struct_meta['rmsd_weights'])
CONTACT_RADIUS    = struct_meta['contact_radius_angstrom']
BURIAL_THRESHOLD  = struct_meta['burial_threshold_contacts']
HEME_FE_AF2       = {k: np.array(v) for k, v in struct_meta['heme_fe_af2'].items()}
INTERFACE_POS     = {k: set(v) for k, v in struct_meta['interface_positions'].items()}

# Load MSA column stats
with open(DATA_DIR / 'msa_col_stats.json') as f:
    MSA_STATS = json.load(f)

BLOSUM62 = substitution_matrices.load('BLOSUM62')
PARSER   = PDBParser(QUIET=True)
print("All artifacts loaded.")

All artifacts loaded.


## Physicochemical tables (from notebook 03)

In [2]:
AA3_TO_1 = {
    'Ala': 'A', 'Arg': 'R', 'Asn': 'N', 'Asp': 'D', 'Cys': 'C',
    'Gln': 'Q', 'Glu': 'E', 'Gly': 'G', 'His': 'H', 'Ile': 'I',
    'Leu': 'L', 'Lys': 'K', 'Met': 'M', 'Phe': 'F', 'Pro': 'P',
    'Ser': 'S', 'Thr': 'T', 'Trp': 'W', 'Tyr': 'Y', 'Val': 'V'
}

AA_VOLUME = {
    'A': 67,  'R': 148, 'N': 96,  'D': 91,  'C': 86,
    'Q': 114, 'E': 109, 'G': 48,  'H': 118, 'I': 124,
    'L': 124, 'K': 135, 'M': 124, 'F': 135, 'P': 90,
    'S': 73,  'T': 93,  'W': 163, 'Y': 141, 'V': 105,
}
AA_CHARGE = {
    'A': 0, 'R': 1, 'N': 0, 'D': -1, 'C': 0, 'Q': 0, 'E': -1,
    'G': 0, 'H': 0, 'I': 0, 'L': 0,  'K': 1, 'M': 0, 'F': 0,
    'P': 0, 'S': 0, 'T': 0, 'W': 0,  'Y': 0, 'V': 0,
}
AA_HYDROPHOB = {
    'A': 1.8, 'R': -4.5, 'N': -3.5, 'D': -3.5, 'C': 2.5, 'Q': -3.5,
    'E': -3.5,'G': -0.4, 'H': -3.2, 'I': 4.5,  'L': 3.8, 'K': -3.9,
    'M': 1.9, 'F': 2.8,  'P': -1.6, 'S': -0.8, 'T': -0.7,'W': -0.9,
    'Y': -1.3,'V': 4.2,
}

UNIPROT_MAP = {'HBB': 'P68871', 'HBA1': 'P69905', 'HBA2': 'P69905'}
AF2_API = 'https://alphafold.ebi.ac.uk/api/prediction'

## Define the 5 Indian variants to score

In [3]:
# These mirror the keys in backend/data/variants.py exactly.
# Fields that are PRESERVED (not replaced by ML) are carried forward as-is.
INDIAN_VARIANTS = [
    {
        'variant_id': 'HBB-Glu6Val',
        'standard_name': 'HBB:p.Glu6Val',
        'common_name': 'Sickle Cell (HbS)',
        'gene': 'HBB', 'position': 6, 'wt_aa': 'Glu', 'mut_aa': 'Val',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.20A>T',
        'population_data': [
            {'state': 'Odisha',          'frequency': 0.33, 'source': 'ICMR NSCAEM 2023'},
            {'state': 'Chhattisgarh',    'frequency': 0.28, 'source': 'ICMR NSCAEM 2023'},
            {'state': 'Maharashtra',     'frequency': 0.15, 'source': 'Colah et al. 2017'},
            {'state': 'Madhya Pradesh',  'frequency': 0.11, 'source': 'ICMR NSCAEM 2023'},
            {'state': 'Jharkhand',       'frequency': 0.08, 'source': 'WHO India 2022'},
        ],
        'plain_language_summary': 'This mutation replaces glutamic acid with valine at position 6 of the beta-globin chain. Under low-oxygen conditions, the mutant hemoglobin (HbS) polymerizes into rigid fibres, deforming red blood cells into a sickle shape. These sickled cells block small blood vessels, causing severe pain crises, organ damage, and reduced lifespan. This is the molecular basis of sickle cell disease — one of the most common serious inherited disorders in India.',
        'clinical_notes': "Homozygous HbSS causes sickle cell disease. Heterozygous HbAS (carrier) is generally asymptomatic. India's National Sickle Cell Elimination Mission targets this variant specifically.",
    },
    {
        'variant_id': 'HBB-Glu26Lys',
        'standard_name': 'HBB:p.Glu26Lys',
        'common_name': 'Hemoglobin E (HbE)',
        'gene': 'HBB', 'position': 26, 'wt_aa': 'Glu', 'mut_aa': 'Lys',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.79G>A',
        'population_data': [
            {'state': 'West Bengal', 'frequency': 0.11, 'source': 'Das et al. 2020'},
            {'state': 'Assam',       'frequency': 0.09, 'source': 'ICMR 2019'},
            {'state': 'Odisha',      'frequency': 0.07, 'source': 'HbVar India entries'},
            {'state': 'Manipur',     'frequency': 0.14, 'source': 'Moirangthem et al. 2014'},
        ],
        'plain_language_summary': 'Hemoglobin E is caused by a mutation at position 26 of the beta-globin chain. It is the second most common structural hemoglobin variant worldwide and very common in Northeast India and Southeast Asia. Heterozygous carriers (HbAE) are usually healthy. When inherited together with beta-thalassemia (HbE/beta-thalassemia), it can cause a moderately severe anemia requiring monitoring and sometimes transfusion.',
        'clinical_notes': 'HbE alone is mild. HbE combined with beta-thalassemia (compound heterozygote) results in a clinically significant thalassemia syndrome. Prevalent in Northeast India, West Bengal, and Assam.',
    },
    {
        'variant_id': 'HBB-Lys17Asn',
        'standard_name': 'HBB:p.Lys17Asn',
        'common_name': None,
        'gene': 'HBB', 'position': 17, 'wt_aa': 'Lys', 'mut_aa': 'Asn',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.51A>C',
        'population_data': [
            {'state': 'Punjab',  'frequency': 0.03, 'source': 'HbVar India entries'},
            {'state': 'Gujarat', 'frequency': 0.02, 'source': 'Colah et al. 2010'},
        ],
        'plain_language_summary': 'This variant at position 17 of the beta-globin chain is located on the outer surface of the protein, far from the oxygen-binding heme pocket. The amino acid change is chemically conservative and does not significantly alter the protein\'s 3D structure or function. This variant is classified as benign — carriers do not develop hemoglobinopathy from this variant alone.',
        'clinical_notes': 'Rare, benign surface variant. No clinical significance as a heterozygote or homozygote. Documented in small numbers in Punjab and Gujarat.',
    },
    {
        'variant_id': 'HBA1-Asp74His',
        'standard_name': 'HBA1:p.Asp74His',
        'common_name': 'Hemoglobin Q-India',
        'gene': 'HBA1', 'position': 74, 'wt_aa': 'Asp', 'mut_aa': 'His',
        'uniprot_id': 'P69905',
        'hgvs': 'NM_000558.5:c.220G>C',
        'population_data': [
            {'state': 'West Bengal',    'frequency': 0.02, 'source': 'Thaker et al. 2024'},
            {'state': 'Andhra Pradesh', 'frequency': 0.01, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'Hemoglobin Q-India is an alpha-globin variant where histidine replaces aspartic acid at position 74 — a position that sits closer to the heme pocket than typical surface variants. The structural analysis shows it sits at the alpha-beta subunit interface, which can affect how the four hemoglobin chains cooperate in binding and releasing oxygen. Heterozygotes are usually mildly affected; coinheritance with alpha-thalassemia can worsen the clinical picture.',
        'clinical_notes': 'Rare variant documented in India, particularly West Bengal. The interface location means it can affect cooperative oxygen binding. Monitor in compound heterozygotes.',
    },
    {
        'variant_id': 'HBB-Val98Met',
        'standard_name': 'HBB:p.Val98Met',
        'common_name': None,
        'gene': 'HBB', 'position': 98, 'wt_aa': 'Val', 'mut_aa': 'Met',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.292G>A',
        'population_data': [
            {'state': 'Tamil Nadu', 'frequency': 0.01, 'source': 'Case report 2021'},
            {'state': 'Karnataka',  'frequency': 0.01, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'This variant at position 98 of beta-globin sits in the heme pocket — the critical oxygen-binding region of the protein. At only 3.1 angstroms from the heme iron, any disruption here directly impairs oxygen binding and release. Despite the methionine substitution appearing conservative by sequence alone (BLOSUM62 score +1), the structural context makes this severely pathogenic. This is an example where sequence-only tools may underestimate severity and structural analysis is critical.',
        'clinical_notes': 'Rare variant with strong structural evidence for pathogenicity. The proximity to heme iron (3.1Å) and high conservation (position entropy 0.05) combined indicate severe impact despite conservative amino acid substitution.',
    },
    # --- 5 additional Indian variants ---
    {
        'variant_id': 'HBB-Glu121Gln',
        'standard_name': 'HBB:p.Glu121Gln',
        'common_name': 'Hemoglobin D-Punjab (HbD)',
        'gene': 'HBB', 'position': 121, 'wt_aa': 'Glu', 'mut_aa': 'Gln',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.364G>C',
        'population_data': [
            {'state': 'Punjab',  'frequency': 0.03, 'source': 'Colah et al. 2010'},
            {'state': 'Gujarat', 'frequency': 0.02, 'source': 'Colah et al. 2010'},
            {'state': 'Sindh region (historical)', 'frequency': 0.04, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'Hemoglobin D-Punjab arises from substitution of glutamine for glutamic acid at position 121 of the beta-globin chain — a surface-exposed helix residue. The charge-neutral substitution (loss of one negative charge) does not disrupt the heme pocket or subunit interface. Heterozygous carriers (HbAD) are clinically normal. However, compound heterozygosity with HbS (HbSD disease) produces a moderate-to-severe sickling disorder because HbD co-polymerizes with HbS under deoxygenated conditions. HbD-Punjab is the most prevalent hemoglobin variant in North India.',
        'clinical_notes': 'HbAD: asymptomatic. HbDD: very mild microcytic anemia. HbSD disease: moderate sickling syndrome requiring monitoring. Prevalent in Punjab, Gujarat, and historically the undivided Punjab/Sindh region.',
    },
    {
        'variant_id': 'HBB-Glu121Lys',
        'standard_name': 'HBB:p.Glu121Lys',
        'common_name': 'Hemoglobin O-Arab (HbO)',
        'gene': 'HBB', 'position': 121, 'wt_aa': 'Glu', 'mut_aa': 'Lys',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.364G>A',
        'population_data': [
            {'state': 'West Bengal', 'frequency': 0.01, 'source': 'Das et al. 2020'},
            {'state': 'Odisha',      'frequency': 0.01, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'Hemoglobin O-Arab replaces the negatively charged glutamic acid with positively charged lysine at beta-globin position 121, reversing the surface charge at that site. Although heterozygous carriers (HbAO) are usually asymptomatic, compound inheritance with HbS (HbSO disease) produces a severe sickling syndrome clinically similar to homozygous sickle cell disease. HbO can also interact with beta-thalassemia mutations, worsening anemia. The variant has been documented in Eastern India and among communities with historical ties to the Arab world.',
        'clinical_notes': 'HbAO: clinically silent. HbSO disease: severe sickling — treat like HbSS. HbO/beta-thalassemia: moderate-to-severe thalassemia syndrome. Documented in West Bengal and Odisha.',
    },
    {
        'variant_id': 'HBA1-Ala120Thr',
        'standard_name': 'HBA1:p.Ala120Thr',
        'common_name': 'Hemoglobin J-Meerut',
        'gene': 'HBA1', 'position': 120, 'wt_aa': 'Ala', 'mut_aa': 'Thr',
        'uniprot_id': 'P69905',
        'hgvs': 'NM_000558.5:c.358G>A',
        'population_data': [
            {'state': 'Uttar Pradesh', 'frequency': 0.01, 'source': 'Sukumaran et al. 1972'},
            {'state': 'Delhi',         'frequency': 0.005, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'Hemoglobin J-Meerut was first identified in a family from Meerut, Uttar Pradesh in 1972, making it one of the earliest alpha-globin variants described in India. The substitution of threonine for alanine at position 120 of the alpha-globin chain occurs at a surface-exposed residue of the GH helix, distant from both the heme pocket and the alpha-beta subunit contact interface. The change is structurally conservative and does not impair oxygen binding or cooperative function. All known carriers have been clinically asymptomatic.',
        'clinical_notes': 'Benign alpha-globin variant. No hematological or clinical consequences in heterozygotes or homozygotes. Historically significant as one of the first Indian hemoglobin variants to be molecularly characterized.',
    },
    {
        'variant_id': 'HBA1-Asp47His',
        'standard_name': 'HBA1:p.Asp47His',
        'common_name': 'Hemoglobin Hasharon',
        'gene': 'HBA1', 'position': 47, 'wt_aa': 'Asp', 'mut_aa': 'His',
        'uniprot_id': 'P69905',
        'hgvs': 'NM_000558.5:c.139G>C',
        'population_data': [
            {'state': 'Gujarat',     'frequency': 0.01, 'source': 'HbVar India entries'},
            {'state': 'Maharashtra', 'frequency': 0.005, 'source': 'Colah et al. 2010'},
        ],
        'plain_language_summary': 'Hemoglobin Hasharon carries a histidine substitution for aspartic acid at position 47 of the alpha-globin chain — a residue in the CD loop near the heme pocket. The replacement of an acidic residue with a neutral imidazole group slightly destabilizes the local structure, reducing the affinity of the mutant chain for heme. This results in a mildly unstable hemoglobin. Most heterozygous carriers have a mild hemolytic anemia that is compensated under normal conditions but can decompensate under oxidative stress (infection, drugs). The variant has been reported in India as well as in populations of Middle Eastern and Mediterranean origin.',
        'clinical_notes': 'Mildly unstable hemoglobin. Heterozygotes may show compensated hemolytic anemia. Avoid oxidant drugs (dapsone, primaquine). Coinheritance with alpha-thalassemia may reduce expression of the abnormal chain.',
    },
    {
        'variant_id': 'HBB-Asn102Thr',
        'standard_name': 'HBB:p.Asn102Thr',
        'common_name': 'Hemoglobin Kansas',
        'gene': 'HBB', 'position': 102, 'wt_aa': 'Asn', 'mut_aa': 'Thr',
        'uniprot_id': 'P68871',
        'hgvs': 'NM_000518.5:c.305A>C',
        'population_data': [
            {'state': 'Tamil Nadu',  'frequency': 0.005, 'source': 'Case report, Venkatesan et al. 2019'},
            {'state': 'Maharashtra', 'frequency': 0.005, 'source': 'HbVar India entries'},
        ],
        'plain_language_summary': 'Hemoglobin Kansas carries a threonine substitution for asparagine at position 102 of the beta-globin chain — a critical residue at the alpha1-beta2 subunit interface involved in the T-to-R quaternary conformational switch during oxygen binding. The substitution dramatically lowers oxygen affinity, causing hemoglobin to release oxygen more readily than normal. This leads to cyanosis (blue discoloration of skin and mucous membranes) because a larger fraction of hemoglobin is in the deoxygenated (T-state) form in arterial blood, even at normal oxygen tensions. Despite looking alarming, carriers are generally healthy because tissue oxygen delivery is actually improved.',
        'clinical_notes': 'Low-oxygen-affinity variant. Presents with cyanosis and apparent polycythemia (compensatory). Not pathological — tissue oxygenation is adequate or improved. Avoidance of unnecessary treatment is key; misdiagnosis as methemoglobinemia has been reported.',
    },
]

print(f"Loaded {len(INDIAN_VARIANTS)} Indian variants to score")

Loaded 10 Indian variants to score


## Load AF2 structures (from notebook 03 cache)

In [4]:
def load_cached_structure(uniprot_id: str):
    path = DATA_DIR / f'AF2_{uniprot_id}.pdb'
    if not path.exists():
        print(f"Downloading AF2 for {uniprot_id}...")
        meta = requests.get(f'{AF2_API}/{uniprot_id}', timeout=15).json()
        pdb_text = requests.get(meta[0]['pdbUrl'], timeout=30).text
        path.write_text(pdb_text)
    return PARSER.get_structure(uniprot_id, str(path))[0]


af2_models = {
    'HBB':  load_cached_structure('P68871'),
    'HBA1': load_cached_structure('P69905'),
}
af2_models['HBA2'] = af2_models['HBA1']
print("AF2 models ready.")

AF2 models ready.


## Fetch SIFT + PolyPhen from Ensembl VEP REST API

In [5]:
VEP_URL = 'https://rest.ensembl.org/vep/human/hgvs'
VEP_HEADERS = {'Content-Type': 'application/json', 'Accept': 'application/json'}


def fetch_vep(hgvs: str) -> dict:
    """
    Query Ensembl VEP REST API for SIFT and PolyPhen2 scores.
    Returns dict with keys: sift_score, sift_prediction, polyphen_score, polyphen_prediction.
    Falls back to None values if the API call fails or annotations are missing.
    """
    result = {'sift_score': None, 'sift_prediction': None,
              'polyphen_score': None, 'polyphen_prediction': None}
    try:
        url = f'{VEP_URL}/{requests.utils.quote(hgvs)}'
        r = requests.get(url, headers=VEP_HEADERS, timeout=20,
                         params={'sift': 1, 'polyphen': 1})
        if r.status_code != 200:
            print(f"  VEP {hgvs}: HTTP {r.status_code}")
            return result
        data = r.json()
        # Navigate to the first transcript consequence
        for entry in data:
            for tc in entry.get('transcript_consequences', []):
                if 'sift_score' in tc:
                    result['sift_score'] = round(tc['sift_score'], 4)
                    pred = tc.get('sift_prediction', '')
                    result['sift_prediction'] = pred.replace('_', ' ').title() if pred else None
                if 'polyphen_score' in tc:
                    result['polyphen_score'] = round(tc['polyphen_score'], 4)
                    pred = tc.get('polyphen_prediction', '')
                    result['polyphen_prediction'] = pred.replace('_', ' ').title() if pred else None
                if result['sift_score'] is not None:
                    break  # first transcript with annotation is sufficient
            if result['sift_score'] is not None:
                break
    except Exception as e:
        print(f"  VEP {hgvs}: {e}")
    return result


print("Fetching VEP annotations (one request per variant, 0.5s between requests)...")
vep_results = {}
for v in INDIAN_VARIANTS:
    print(f"  {v['variant_id']} ({v['hgvs']})")
    vep_results[v['variant_id']] = fetch_vep(v['hgvs'])
    time.sleep(0.5)

for vid, r in vep_results.items():
    print(f"  {vid}: SIFT={r['sift_score']} PolyPhen={r['polyphen_score']}")

Fetching VEP annotations (one request per variant, 0.5s between requests)...
  HBB-Glu6Val (NM_000518.5:c.20A>T)


  HBB-Glu26Lys (NM_000518.5:c.79G>A)


  HBB-Lys17Asn (NM_000518.5:c.51A>C)


  VEP NM_000518.5:c.51A>C: HTTP 400


  HBA1-Asp74His (NM_000558.5:c.220G>C)


...

  VEP NM_000558.5:c.358G>A: HTTP 400


  HBA1-Asp47His (NM_000558.5:c.139G>C)


  VEP NM_000558.5:c.139G>C: HTTP 400


  HBB-Asn102Thr (NM_000518.5:c.305A>C)


  HBB-Glu6Val: SIFT=0 PolyPhen=None
  HBB-Glu26Lys: SIFT=0.01 PolyPhen=None
  HBB-Lys17Asn: SIFT=None PolyPhen=None
  HBA1-Asp74His: SIFT=1 PolyPhen=None
  HBB-Val98Met: SIFT=None PolyPhen=None
  HBB-Glu121Gln: SIFT=0.03 PolyPhen=None
  HBB-Glu121Lys: SIFT=0 PolyPhen=None
  HBA1-Ala120Thr: SIFT=None PolyPhen=None
  HBA1-Asp47His: SIFT=None PolyPhen=None
  HBB-Asn102Thr: SIFT=0 PolyPhen=None


## Compute all features + classifier prediction per variant

In [6]:
STANDARD_AAS = set('ACDEFGHIKLMNPQRSTVWY')


def compute_features(v: dict, af2_model) -> dict:
    """
    Compute all 9 training features + extra report fields for one variant.
    Returns a dict matching the SequenceFeatures + StructuralFeatures shape.
    """
    gene = v['gene']
    pos  = v['position']
    wt3  = v['wt_aa']
    mut3 = v['mut_aa']
    wt1  = AA3_TO_1[wt3]
    mut1 = AA3_TO_1[mut3]

    # --- Sequence features ---
    try:
        blosum62 = int(BLOSUM62[wt1][mut1])
    except KeyError:
        blosum62 = -4

    msa_gene = gene if gene in MSA_STATS else 'HBA1'
    col_data  = MSA_STATS.get(msa_gene, {}).get(str(pos), {})
    conservation  = col_data.get('conservation', 0.5)
    position_entropy = col_data.get('entropy', math.log2(20) / 2)
    pssm_score    = col_data.get('pssm', {}).get(mut1, 0.0)

    # --- Structural features ---
    residue = None
    try:
        residue = af2_model['A'][(' ', pos, ' ')]
    except KeyError:
        pass

    ca = np.array(residue['CA'].get_coord()) if (residue and 'CA' in residue) else None
    plddt = round(float(residue['CA'].get_bfactor()), 1) if (residue and 'CA' in residue) else 70.0

    heme_fe  = HEME_FE_AF2.get(gene)
    heme_dist = round(float(np.linalg.norm(ca - heme_fe)), 2) if (ca is not None and heme_fe is not None) else 20.0

    contact_count = 0
    if ca is not None:
        for res in af2_model['A'].get_residues():
            if res.get_id()[0] != ' ':
                continue
            if 'CA' in res:
                d = np.linalg.norm(ca - np.array(res['CA'].get_coord()))
                if 0.5 < d <= CONTACT_RADIUS:
                    contact_count += 1

    buried   = contact_count >= BURIAL_THRESHOLD
    exposure = 'buried' if buried else 'surface'
    is_interface = pos in INTERFACE_POS.get(gene, set())

    avg_vol = 105.0
    vol_delta = abs(AA_VOLUME.get(wt1, avg_vol) - AA_VOLUME.get(mut1, avg_vol))
    contact_map_delta = round((vol_delta / avg_vol) * contact_count, 2)

    # RMSD proxy
    dc = abs(AA_CHARGE.get(wt1, 0) - AA_CHARGE.get(mut1, 0))
    dv = abs(AA_VOLUME.get(wt1, 100) - AA_VOLUME.get(mut1, 100)) / 100.0
    dh = abs(AA_HYDROPHOB.get(wt1, 0) - AA_HYDROPHOB.get(mut1, 0)) if buried else 0.0
    rmsd = round(float(np.array([dc, dv, dh]) @ RMSD_WEIGHTS), 2)

    return {
        # Sequence
        'blosum62_score': blosum62,
        'conservation_score': round(conservation, 4),
        'pssm_score': round(pssm_score, 4),
        'position_entropy': round(position_entropy, 4),
        # Structural
        'af2_confidence': plddt,
        'heme_distance_angstrom': heme_dist,
        'burial_binary': int(buried),
        'residue_exposure': exposure,
        'contact_count_wt': contact_count,
        'is_interface_residue': is_interface,
        'contact_map_delta': contact_map_delta,
        'rmsd_angstrom': rmsd,
    }


print("Computing features for 5 Indian variants...")
feature_rows = {}
for v in INDIAN_VARIANTS:
    model = af2_models.get(v['gene'])
    feature_rows[v['variant_id']] = compute_features(v, model)
    print(f"  {v['variant_id']}: heme_dist={feature_rows[v['variant_id']]['heme_distance_angstrom']}Å  "
          f"burial={feature_rows[v['variant_id']]['residue_exposure']}")

Computing features for 5 Indian variants...
  HBB-Glu6Val: heme_dist=38.06Å  burial=surface
  HBB-Glu26Lys: heme_dist=27.77Å  burial=buried
  HBB-Lys17Asn: heme_dist=36.96Å  burial=surface
  HBA1-Asp74His: heme_dist=108.3Å  burial=surface
  HBB-Val98Met: heme_dist=6.71Å  burial=surface
  HBB-Glu121Gln: heme_dist=42.1Å  burial=surface
  HBB-Glu121Lys: heme_dist=42.1Å  burial=surface
  HBA1-Ala120Thr: heme_dist=93.88Å  burial=surface
  HBA1-Asp47His: heme_dist=127.21Å  burial=surface
  HBB-Asn102Thr: heme_dist=13.32Å  burial=surface


## Run classifier

In [7]:
predictions = {}
for v in INDIAN_VARIANTS:
    vid = v['variant_id']
    feats = feature_rows[vid]
    X_vec = np.array([[feats[c] for c in FEATURE_COLS]])
    X_imp = imputer.transform(X_vec)

    proba = clf.predict_proba(X_imp)[0]
    pred_idx = np.argmax(proba)
    severity = le.inverse_transform([pred_idx])[0]
    confidence = round(float(proba[pred_idx]), 4)

    predictions[vid] = {'severity': severity, 'confidence': confidence, 'proba': dict(zip(le.classes_, proba.round(4)))}
    print(f"  {vid}: {severity} (confidence={confidence:.3f})  | proba={predictions[vid]['proba']}")

  HBB-Glu6Val: mild (confidence=0.864)  | proba={'benign': np.float64(0.0731), 'mild': np.float64(0.8642), 'severe': np.float64(0.0627)}


  HBB-Glu26Lys: severe (confidence=0.694)  | proba={'benign': np.float64(0.016), 'mild': np.float64(0.2899), 'severe': np.float64(0.6941)}


  HBB-Lys17Asn: mild (confidence=0.813)  | proba={'benign': np.float64(0.1032), 'mild': np.float64(0.8129), 'severe': np.float64(0.084)}
  HBA1-Asp74His: mild (confidence=0.902)  | proba={'benign': np.float64(0.0984), 'mild': np.float64(0.9016), 'severe': np.float64(0.0)}


  HBB-Val98Met: mild (confidence=0.582)  | proba={'benign': np.float64(0.0318), 'mild': np.float64(0.5818), 'severe': np.float64(0.3864)}


  HBB-Glu121Gln: benign (confidence=0.526)  | proba={'benign': np.float64(0.5265), 'mild': np.float64(0.4735), 'severe': np.float64(0.0)}
  HBB-Glu121Lys: benign (confidence=0.513)  | proba={'benign': np.float64(0.5129), 'mild': np.float64(0.4871), 'severe': np.float64(0.0)}


  HBA1-Ala120Thr: mild (confidence=0.663)  | proba={'benign': np.float64(0.1135), 'mild': np.float64(0.6632), 'severe': np.float64(0.2233)}


  HBA1-Asp47His: mild (confidence=0.847)  | proba={'benign': np.float64(0.1193), 'mild': np.float64(0.8468), 'severe': np.float64(0.0339)}
  HBB-Asn102Thr: mild (confidence=0.735)  | proba={'benign': np.float64(0.0591), 'mild': np.float64(0.735), 'severe': np.float64(0.2058)}


## Sanity checks

In [8]:
expected = {
    # Original 5 Indian variants
    'HBB-Glu6Val':  'severe',   # HbS — sickling disease
    'HBB-Glu26Lys': 'mild',     # HbE — mild as heterozygote
    'HBB-Lys17Asn': 'benign',   # surface variant, no clinical impact
    'HBA1-Asp74His': 'mild',    # Hb Q-India — interface, mildly significant
    'HBB-Val98Met': 'severe',   # heme-pocket variant, severely pathogenic
    # 5 new Indian variants
    'HBB-Glu121Gln': 'mild',    # HbD-Punjab — asymptomatic heterozygote, mild in compound
    'HBB-Glu121Lys': 'mild',    # HbO-Arab — asymptomatic heterozygote (severe only in HbSO)
    'HBA1-Ala120Thr': 'benign', # Hb J-Meerut — benign, all carriers asymptomatic
    'HBA1-Asp47His': 'mild',    # Hb Hasharon — mild hemolytic anemia, unstable
    'HBB-Asn102Thr': 'mild',    # Hb Kansas — low O2 affinity, cyanosis but not truly pathological
}

print("Sanity check (expected vs ML predicted):")
all_pass = True
for vid, exp_sev in expected.items():
    pred_sev = predictions[vid]['severity']
    status = '✓' if pred_sev == exp_sev else '✗  MISMATCH'
    print(f"  {vid}: expected={exp_sev}  predicted={pred_sev}  {status}")
    if pred_sev != exp_sev:
        all_pass = False

if not all_pass:
    print("\nOne or more predictions differ from expected clinical severity.")
    print("This is informative rather than necessarily wrong — check feature values above.")

# Val98Met must have smallest heme distance among HBB variants
heme_dists = {v['variant_id']: feature_rows[v['variant_id']]['heme_distance_angstrom'] for v in INDIAN_VARIANTS}
min_vid = min(heme_dists, key=heme_dists.get)
print(f"\nSmallest heme distance: {min_vid} at {heme_dists[min_vid]}Å  "
      f"({'✓ expected' if min_vid == 'HBB-Val98Met' else '✗ unexpected — check structure'})")
print(f"\nAll heme distances:")
for vid, dist in sorted(heme_dists.items(), key=lambda x: x[1]):
    print(f"  {vid}: {dist}Å")

Sanity check (expected vs ML predicted):
  HBB-Glu6Val: expected=severe  predicted=mild  ✗  MISMATCH
  HBB-Glu26Lys: expected=mild  predicted=severe  ✗  MISMATCH
  HBB-Lys17Asn: expected=benign  predicted=mild  ✗  MISMATCH
  HBA1-Asp74His: expected=mild  predicted=mild  ✓
  HBB-Val98Met: expected=severe  predicted=mild  ✗  MISMATCH
  HBB-Glu121Gln: expected=mild  predicted=benign  ✗  MISMATCH
  HBB-Glu121Lys: expected=mild  predicted=benign  ✗  MISMATCH
  HBA1-Ala120Thr: expected=benign  predicted=mild  ✗  MISMATCH
  HBA1-Asp47His: expected=mild  predicted=mild  ✓
  HBB-Asn102Thr: expected=mild  predicted=mild  ✓

One or more predictions differ from expected clinical severity.
This is informative rather than necessarily wrong — check feature values above.

Smallest heme distance: HBB-Val98Met at 6.71Å  (✓ expected)

All heme distances:
  HBB-Val98Met: 6.71Å
  HBB-Asn102Thr: 13.32Å
  HBB-Glu26Lys: 27.77Å
  HBB-Lys17Asn: 36.96Å
  HBB-Glu6Val: 38.06Å
  HBB-Glu121Gln: 42.1Å
  HBB-Glu121Lys

## Render and write variants_ml.py

In [9]:
def render_variant_dict(v: dict, feats: dict, pred: dict, vep: dict) -> str:
    """Render one VARIANT_DB entry as a Python literal string."""
    vid        = v['variant_id']
    severity   = pred['severity']
    confidence = pred['confidence']
    cn = f'"{v["common_name"]}"' if v['common_name'] else 'None'

    sift_score = vep.get('sift_score')
    sift_pred  = vep.get('sift_prediction')
    pp_score   = vep.get('polyphen_score')
    pp_pred    = vep.get('polyphen_prediction')

    pop_lines = ',\n            '.join(
        f'{{"state": "{p["state"]}", "frequency": {p["frequency"]}, "source": "{p["source"]}"}}'
        for p in v['population_data']
    )

    summary  = v['plain_language_summary'].replace('"', '\\"')
    clin_raw = (v.get('clinical_notes') or '').replace('"', '\\"')
    clin_line = f'"clinical_notes": "{clin_raw}"' if clin_raw else '"clinical_notes": None'

    exposure     = feats['residue_exposure']
    is_interface = str(feats['is_interface_residue'])

    sift_score_str = str(sift_score) if sift_score is not None else 'None'
    sift_pred_str  = f'"{sift_pred}"' if sift_pred else 'None'
    pp_score_str   = str(pp_score)   if pp_score  is not None else 'None'
    pp_pred_str    = f'"{pp_pred}"'  if pp_pred   else 'None'

    lines = [
        f'    "{vid}": {{',
        f'        "variant_id": "{vid}",',
        f'        "standard_name": "{v["standard_name"]}",',
        f'        "common_name": {cn},',
        f'        "gene": "{v["gene"]}",',
        f'        "position": {v["position"]},',
        f'        "wildtype_aa": "{v["wt_aa"]}",',
        f'        "mutant_aa": "{v["mut_aa"]}",',
        f'        "severity": "{severity}",',
        f'        "confidence": {confidence},',
        f'        "sequence_features": {{',
        f'            "blosum62_score": {feats["blosum62_score"]},',
        f'            "conservation_score": {feats["conservation_score"]},',
        f'            "pssm_score": {feats["pssm_score"]},',
        f'            "position_entropy": {feats["position_entropy"]}',
        f'        }},',
        f'        "structural_features": {{',
        f'            "rmsd_angstrom": {feats["rmsd_angstrom"]},',
        f'            "heme_distance_angstrom": {feats["heme_distance_angstrom"]},',
        f'            "residue_exposure": "{exposure}",',
        f'            "is_interface_residue": {is_interface},',
        f'            "contact_map_delta": {feats["contact_map_delta"]},',
        f'            "af2_confidence": {feats["af2_confidence"]}',
        f'        }},',
        f'        "tool_comparison": {{',
        f'            "sift_score": {sift_score_str},',
        f'            "sift_prediction": {sift_pred_str},',
        f'            "polyphen_score": {pp_score_str},',
        f'            "polyphen_prediction": {pp_pred_str},',
        f'            "hemoscope_confidence": {confidence},',
        f'            "hemoscope_prediction": "{severity}"',
        f'        }},',
        f'        "population_data": [',
        f'            {pop_lines}',
        f'        ],',
        f'        "plain_language_summary": "{summary}",',
        f'        {clin_line},',
        f'        "uniprot_id": "{v["uniprot_id"]}"',
        f'    }}',
    ]
    return '\n'.join(lines)


variant_entries = []
for v in INDIAN_VARIANTS:
    vid = v['variant_id']
    entry = render_variant_dict(v, feature_rows[vid], predictions[vid], vep_results[vid])
    variant_entries.append(entry)

search_index_lines = []
for v in INDIAN_VARIANTS:
    vid = v['variant_id']
    cn  = f'"{v["common_name"]}"' if v['common_name'] else 'None'
    sev = predictions[vid]['severity']
    search_index_lines.append(
        f'    {{"variant_id": "{vid}", "standard_name": "{v["standard_name"]}", '
        f'"common_name": {cn}, "gene": "{v["gene"]}", "severity": "{sev}"}}'
    )

with open(MODELS_DIR / 'cv_scores.json') as f:
    cv_scores = json.load(f)

# Pre-compute joined blocks so the final string can use simple variable refs
variant_block      = ',\n'.join(variant_entries)
search_index_block = ',\n'.join(search_index_lines)
f1_mean = cv_scores['rf_macro_f1_mean']
f1_std  = cv_scores['rf_macro_f1_std']
n_train = cv_scores['n_training_variants']

output = (
    '"""\n'
    'ML-generated variant database — produced by notebooks/05_generate_variants_py.ipynb\n\n'
    'Feature pipeline:\n'
    '  - Sequence: BLOSUM62 + conservation/PSSM/entropy from UniProt globin homologs (Biopython)\n'
    '  - Structure: AlphaFold2 EBI API (pLDDT, heme distance, burial, contacts)\n'
    '  - RMSD: calibrated proxy — see notebooks/03_structural_features.ipynb\n'
    f'  - Classifier: CalibratedRandomForestClassifier, 5-fold CV macro-F1={f1_mean:.3f}±{f1_std:.3f}\n'
    f'  - Trained on {n_train} ClinVar HBB/HBA1 variants\n\n'
    'To update: re-run notebooks 01-05 in order, then replace backend/data/variants.py with this file.\n'
    '"""\n\n'
    'VARIANT_DB = {\n'
    + variant_block + '\n'
    '}\n\n'
    'SEARCH_INDEX = [\n'
    + search_index_block + '\n'
    ']\n'
)

out_path = BACKEND_DATA / 'variants_ml.py'
out_path.write_text(output)
print(f"Written → {out_path}")
print(f"\nFirst 800 chars:\n{output[:800]}")

Written → ../backend/data/variants_ml.py

First 800 chars:
"""
ML-generated variant database — produced by notebooks/05_generate_variants_py.ipynb

Feature pipeline:
  - Sequence: BLOSUM62 + conservation/PSSM/entropy from UniProt globin homologs (Biopython)
  - Structure: AlphaFold2 EBI API (pLDDT, heme distance, burial, contacts)
  - RMSD: calibrated proxy — see notebooks/03_structural_features.ipynb
  - Classifier: CalibratedRandomForestClassifier, 5-fold CV macro-F1=0.500±0.162
  - Trained on 153 ClinVar HBB/HBA1 variants

To update: re-run notebooks 01-05 in order, then replace backend/data/variants.py with this file.
"""

VARIANT_DB = {
    "HBB-Glu6Val": {
        "variant_id": "HBB-Glu6Val",
        "standard_name": "HBB:p.Glu6Val",
        "common_name": "Sickle Cell (HbS)",
        "gene": "HBB",
        "position": 6,
        "wildtype_a


## Schema validation — load into FastAPI Pydantic models

In [10]:
import sys
sys.path.insert(0, str(Path('../backend').resolve()))

from models import VariantReport
from data.variants_ml import VARIANT_DB as ML_DB

print("Validating schema for all ML variants...")
for k, v in ML_DB.items():
    try:
        VariantReport(**v)
        print(f"  ✓ {k}")
    except Exception as e:
        print(f"  ✗ {k}: {e}")

print("\nDone. Review variants_ml.py, then replace variants.py when satisfied.")

Validating schema for all ML variants...
  ✓ HBB-Glu6Val
  ✓ HBB-Glu26Lys
  ✓ HBB-Lys17Asn
  ✓ HBA1-Asp74His
  ✓ HBB-Val98Met
  ✓ HBB-Glu121Gln
  ✓ HBB-Glu121Lys
  ✓ HBA1-Ala120Thr
  ✓ HBA1-Asp47His
  ✓ HBB-Asn102Thr

Done. Review variants_ml.py, then replace variants.py when satisfied.
